In [ ]:
from __future__ import annotations

import json
from pathlib import Path


# Repo root: parent of `notebooks/` when this notebook lives under `notebooks/`
def repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "notebooks" else cwd

In [ ]:
TEMPO_BPM = 60  # change this to experiment
TIME_STEP_SECONDS = 0.5
# One lyric per time frame; "" = no lyric on that frame.
LYRICS = ["Ho", "la", "", "dron", "de", "", "", "luz"]

In [ ]:
# The matrix logic lives in the package so the notebook, the FastAPI
# endpoint and the regenerated example-scores.json stay in sync.
from aitu_backend.sequence import (
    build_grand_piano_rows,
    parse_timeframe_notes,
    sequence_to_score,
    sequence_to_sparse_payload,
)

# Notation reminder:
#   "*Note"  -> ONSET   (the key is struck)
#   "Note"   -> SUSTAIN  (same key keeps sounding, not struck again)
#   "A || B" -> simultaneous notes in one frame
#   ""       -> a silent frame
#
# Onset rule: a sustain only continues through frames with NO new onset.
# As soon as a frame contains any onset, every carried sustain ends at the
# previous frame (the builder normalizes input to honor this).

ROWS = build_grand_piano_rows()
print(ROWS[0], "...", ROWS[-1], "| count:", len(ROWS))
print(parse_timeframe_notes("*Do-4 || Mi-4"))

In [ ]:
# Frame 0 is a struck chord; Re-4 is struck then held (one 2-frame note);
# Mi-4 is struck then held for 4 frames (a half note); Fa#-5 is struck.
sequence = [
    "*Do-4 || *Mi-4",
    "*Re-4",
    "Re-4",
    "*Mi-4",
    "Mi-4",
    "Mi-4",
    "Mi-4",
    "*Fa#-5",
]

payload = sequence_to_sparse_payload(sequence)
print("shape:", payload["shape"])
print("rows :", payload["rows"])
print("cols :", payload["cols"])
print("onset:", payload["onset"])

In [ ]:
# Human-readable view: each active cell as (col, note, ONSET|sustain).
for r, c, o in zip(payload["rows"], payload["cols"], payload["onset"]):
    kind = "ONSET " if o != -1 else "sustain"
    print(f"col {c:>2}  {ROWS[r]:<6}  {kind}")

# Sparse Representation

The payload sends three parallel arrays sorted by `(col, row)`:

- `rows[i]` / `cols[i]` — an active cell.
- `onset[i]` — `rows[i]` if the cell is a **struck onset**, or `-1` if it is a
  **sustain** (the note keeps sounding without being struck again).

`onset` is what removes the old ambiguity: four `Re-4` cells in a row are
*four eighth notes* when each is an onset, but *one long note* when only the
first is an onset and the rest are `-1` sustains.

In [ ]:
# Quick grid view of just the active rows (O = onset, . = sustain).
active = sorted({r for r in payload["rows"]})
n_cols = payload["shape"][1]
cell = {(c, r): o for r, c, o in zip(payload["rows"], payload["cols"], payload["onset"])}

for r in active:
    line = []
    for c in range(n_cols):
        if (c, r) not in cell:
            line.append(" ")
        elif cell[(c, r)] == -1:
            line.append(".")
        else:
            line.append("O")
    print(f"{ROWS[r]:<6} |{''.join(line)}|")

In [ ]:
scores = [
    sequence_to_score(
        sequence,
        tempo_bpm=TEMPO_BPM,
        time_step_seconds=TIME_STEP_SECONDS,
        title="Grand piano 88-key onset example",
        lyrics=LYRICS,
    )
]

# Persist for the FastAPI `/scores` endpoint (drop None fields for a clean file).
scores_path = repo_root() / "data" / "example-scores.json"
clean = [{k: v for k, v in s.items() if v is not None} for s in scores]
scores_path.write_text(json.dumps(clean, indent=2) + "\n", encoding="utf-8")
print("wrote", scores_path)

In [ ]:
scores[0]